<a href="https://colab.research.google.com/github/Loading-my-brain/GEOG761_Remote_Sensing/blob/main/kaikoura_quake_Pre_Post_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Our usual set up routine
!pip install geemap --quiet

import geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans

import ee
ee.Authenticate()
ee.Initialize(project='x-band-508903') #<- RE: Remember to change this to your own project's name!
                                            # We have a shared project: x-band-508903

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.1 MB/s eta 0:00:00


In [ ]:
"""
Kaikōura Earthquake SAR Change Detection and Blockage Identification

This script utilizes Google Earth Engine and Sentinel-1 Synthetic Aperture Radar (SAR)
imagery to detect land blockages, landslides, and debris deposits resulting from the
Magnitude 7.8 Kaikōura earthquake on November 14, 2016.

Methodology & Date Filters:
- Pre-Earthquake Baseline: 2016-09-01 to 2016-11-13.
- Post-Earthquake Window: 2016-11-14 to 2017-01-31.
- Time Windows & Median Compositing: Rather than using a single date, these multi-month
  windows calculate the temporal median of all satellite passes. This is standard SAR
  practice because it:
    1. Reduces Speckle Noise: Smooths out the random static inherent to raw radar data.
    2. Ensures Complete Coverage: Stitches together multiple sparse passes to guarantee
       the entire bounding box is mapped without gaps.
    3. Filters Weather Anomalies: Removes temporary signal spikes from soil moisture
       (e.g., heavy rain on a single day) to establish a stable, true baseline condition.
- Orbit Matching: Strictly filters for DESCENDING passes to ensure identical viewing
  geometries. This prevents false-positive changes caused by opposite radar look angles
  in steep topographic terrain.
- Masking: The Hansen Global Forest Change dataset is applied to mask out the ocean and
  large water bodies, isolating the analysis strictly to landmasses.
- Thresholding: Severe structural changes are isolated using a +/- 3.0 dB difference threshold.

Outputs:
1. kaikoura_blockages_clean.png
   A transparent PNG overlay highlighting isolated, extreme terrain changes.
   - Red Pixels (<= -3.0 dB): Indicate a significant decrease in radar backscatter
     (e.g., landslide scarps, vegetation removal, or newly pooled water behind debris dams).
   - Yellow Pixels (>= +3.0 dB): Indicate a significant increase in radar backscatter
     (e.g., rockfall, heavy rubble, or debris deposits facing the radar sensor).

2. kaikoura_continuous_change_clean.png
   A continuous difference map (Post minus Pre) across the entire landmass showing
   un-thresholded backscatter shifts.
   - Red: Negative change.
   - White: Minimal to no change.
   - Blue: Positive change.
"""
import ee
import geemap

# Authenticate and Initialize
try:
    ee.Initialize(project='geog-761-neeravh')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='geog-761-neeravh')

# Define AOI
aoi = ee.Geometry.Point([173.68, -42.40]).buffer(50000).bounds()

# Date ranges for strict Descending match
pre_start, pre_end = '2016-09-01', '2016-11-13'
post_start, post_end = '2016-11-14', '2017-01-31'

# Base Collection filtered for DESCENDING passes
s1_descending = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filterBounds(aoi)
                 .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                 .filter(ee.Filter.eq('instrumentMode', 'IW'))
                 .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')))

# Pre and Post composites
pre_desc = s1_descending.filterDate(pre_start, pre_end).select('VV').median().clip(aoi)
post_desc = s1_descending.filterDate(post_start, post_end).select('VV').median().clip(aoi)

# 1. Calculate the Difference (Post - Pre)
sar_difference = post_desc.subtract(pre_desc)

# 2. Mask out the ocean and large water bodies
land_mask = ee.Image('UMD/hansen/global_forest_change_2022_v1_10').select('datamask').eq(1)
land_difference = sar_difference.updateMask(land_mask)

# 3. Isolate extreme changes indicative of landslides and blockages
severe_decrease = land_difference.lte(-3.0).selfMask()
severe_increase = land_difference.gte(3.0).selfMask()

# Convert the thresholded change layers into visual RGB images
decrease_vis = severe_decrease.visualize(palette=['red'])
increase_vis = severe_increase.visualize(palette=['yellow'])

# Blend the red (decrease) and yellow (increase) layers into a single image
blockage_composite = ee.ImageCollection([decrease_vis, increase_vis]).mosaic()

# Define parameters for continuous change
diff_params = {'min': -4, 'max': 4, 'palette': ['red', 'white', 'blue']}

# Save the isolated blockages as a PNG with a transparent background
geemap.get_image_thumbnail(
    blockage_composite,
    'kaikoura_blockages_clean.png',
    vis_params={},  # Empty dict fixes the missing argument error
    dimensions=1000,
    region=aoi,
    format='png'
)

# Save the continuous red/white/blue difference map as well
geemap.get_image_thumbnail(
    land_difference,
    'kaikoura_continuous_change_clean.png',
    vis_params=diff_params,
    dimensions=1000,
    region=aoi,
    format='png'
)

print("Export complete! Please check your file explorer for the new files.")

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2022_v1_10! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by UMD/hansen/global_forest_change_2025_v1_13

Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2022_v1_10

  warnings.warn(warning, category=DeprecationWarning)


EEException: Caller does not have required permission to use project geog-761-neeravh. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=geog-761-neeravh and then retry. Propagation of the new permission may take a few minutes.